# Даты договоров: ЦФТ, отчёт коллег за сентябрь и Альфа

- **ЦФТ** - `CFT_clients_010926.xlsx` (лежит рядом с тетрадкой): дата начала, окончания и закрытия договора.
- **Отчёт за сентябрь** - `Data/09_*.xlsx`. Колонки с датами находятся автоматически (по названию и по содержимому), список печатается. Если найдено не то - задать `EX_DATE_COLS`.
- **Альфа** - моя выгрузка на 01.09: дата открытия и закрытия договора.

Договоры отчёта и ЦФТ сопоставляются по номеру, оставшиеся - по ID (ID договора в отчёте и `id` в ЦФТ). Для каждой пары дат: у скольких договоров заполнены обе, сколько совпадают точно и с точностью до дня, медиана разницы в днях.

Запросов к озеру нет.


In [ ]:
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.max_colwidth', 80)

AS_OF_DATE = '2026-09-01'
BASE_DIR = Path('/home/jovyan/documents/Equaring')
DATA_DIR = BASE_DIR / 'Data'
CFT_FILE = Path('CFT_clients_010926.xlsx').resolve()  # файл лежит рядом с тетрадкой
CFT_SHEET = 0
EX_FILE = None  # None - взять файл 09_*2026*.xlsx из DATA_DIR
EX_SHEET = 0
MINE_FILE = BASE_DIR / ('trade_acquiring_clients_%s.xlsx' % AS_OF_DATE)
OUT_FILE = BASE_DIR / ('te_dates_cft_report_%s.xlsx' % AS_OF_DATE)

# Колонки отчёта. None - найти автоматически
EX_ID_COL = None
EX_NUM_COL = None
EX_DATE_COLS = None  # например ['Дата открытия договора', 'Дата закрытия договора']

if EX_FILE is None:
    cands = sorted(DATA_DIR.glob('09_*2026*.xlsx'))
    print('Кандидаты на отчёт за сентябрь:', [p.name for p in cands])
    EX_FILE = cands[0] if cands else DATA_DIR / '09_Сентябрь_2026.xlsx'
as_of = pd.Timestamp(AS_OF_DATE)
as_of_ru = as_of.strftime('%d.%m.%Y')
MINE_SHEET = 'С ТЭ на %s' % as_of_ru
for p in (CFT_FILE, EX_FILE, MINE_FILE):
    print('есть ' if p.exists() else 'НЕТ  ', p)


## 1. Загрузка

In [ ]:
EMPTY = ('', 'NONE', 'NAN', '<NA>', 'NAT', 'NULL')
DATE_WORDS = ('дата', 'date', 'открыт', 'закрыт', 'окончан', 'начал', 'заключ', 'расторж', 'подключ')

def _low(v):
    return re.sub(r'\s+', ' ', str(v)).strip().lower()

def norm_agr(v):
    if v is None or pd.isna(v):
        return None
    s = str(v).strip()
    if re.fullmatch(r'\d+\.0+', s):
        s = s.split('.')[0]
    s = re.sub(r'\s+', '', s).upper()
    return None if s in EMPTY else s

def norm_id(v):
    k = norm_agr(v)
    if not k:
        return None
    return (k.lstrip('0') or '0') if k.isdigit() else k

def to_date(s):
    s = pd.Series(s, dtype=object)
    iso = s.map(lambda v: isinstance(v, str) and bool(re.match(r'\s*\d{4}-\d{2}-\d{2}', v)))
    out = pd.to_datetime(s.where(~iso), errors='coerce', dayfirst=True)
    out[iso] = pd.to_datetime(s[iso].str.strip().str[:10], errors='coerce', format='%Y-%m-%d')
    out = out.dt.normalize()
    return out.where((out.dt.year >= 1990) & (out.dt.year <= 2200))

def date_share(s):
    filled = s.notna().sum()
    if not filled:
        return 0.0
    digits_only = s.dropna().map(lambda x: bool(re.fullmatch(r'\d+(\.0+)?', str(x).strip())))
    if digits_only.mean() > 0.5:
        return 0.0
    return float(to_date(s).notna().sum() / filled)

def read_with_header(path, sheet, need=('договор',), max_rows=20):
    raw = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    for i in range(min(max_rows, len(raw))):
        cells = ' | '.join(_low(v) for v in raw.iloc[i] if pd.notna(v))
        if all(k in cells for k in need):
            names, seen = [], {}
            for j, v in enumerate(raw.iloc[i]):
                n = re.sub(r'\s+', ' ', str(v)).strip() if pd.notna(v) else 'col_%s' % j
                seen[n] = seen.get(n, 0) + 1
                names.append(n if seen[n] == 1 else '%s_%s' % (n, seen[n]))
            df = raw.iloc[i + 1:].copy()
            df.columns = names
            return df.dropna(how='all').reset_index(drop=True), i
    raise ValueError('Не нашёл строку заголовков с %s в первых %s строках %s' % (need, max_rows, path))

def pick(cols, exact, contains=None, exclude=()):
    low = {c: _low(c) for c in cols}
    for e in exact:
        for c in cols:
            if low[c] == e:
                return c
    if contains:
        for c in cols:
            if all(k in low[c] for k in contains) and not any(x in low[c] for x in exclude):
                return c
    return None

def col_or_none(df, col, fn=None):
    if col is None:
        return pd.Series(None, index=df.index, dtype=object)
    s = df[col]
    return s.map(fn) if fn else s.astype('string').str.strip()


In [ ]:
cft_raw, _ = read_with_header(CFT_FILE, CFT_SHEET)
ccols = list(cft_raw.columns)
C = {
    'agr': pick(ccols, ['номер договора'], ('номер', 'договор'), exclude=('тариф', 'индивид')),
    'name': pick(ccols, ['наименование клиента'], ('наимен',), exclude=('тариф',)),
    'status': pick(ccols, ['статус'], ('статус',)),
    'start': pick(ccols, ['дата начала действия договора'], ('начал',)),
    'end': pick(ccols, ['дата окончания действия договора'], ('окончан',), exclude=('тариф',)),
    'close': pick(ccols, ['дата закрытия'], ('закрыт', 'дата')),
    'id': pick(ccols, ['id']),
}
print('Колонки ЦФТ:', C)
cft = pd.DataFrame({
    'num_key': cft_raw[C['agr']].map(norm_agr),
    'cft_id': col_or_none(cft_raw, C['id'], norm_id),
    'Номер договора (ЦФТ)': col_or_none(cft_raw, C['agr']),
    'Клиент (ЦФТ)': col_or_none(cft_raw, C['name']),
    'Статус (ЦФТ)': col_or_none(cft_raw, C['status']),
    'ЦФТ: начало': to_date(cft_raw[C['start']]),
    'ЦФТ: окончание': to_date(cft_raw[C['end']]),
    'ЦФТ: закрытие': to_date(cft_raw[C['close']]),
})
cft = cft[cft['num_key'].notna()].drop_duplicates('num_key').reset_index(drop=True)
CFT_DATES = ['ЦФТ: начало', 'ЦФТ: окончание', 'ЦФТ: закрытие']
print('ЦФТ: договоров', len(cft))

rep_raw, rep_hdr = read_with_header(EX_FILE, EX_SHEET, need=('инн', 'договор'))
cols = list(rep_raw.columns)
ex_id = EX_ID_COL or pick(cols, ['id договора', 'agr_id', 'abs_agr_id'], ('id', 'договор'))
ex_num = EX_NUM_COL or pick(cols, ['номер договора', '№ договора'], ('номер', 'договор'), exclude=('id',))
if ex_num is None:
    raise KeyError('В отчёте не нашлась колонка номера договора. Задайте EX_NUM_COL. Колонки: %s' % cols)

rep_fill = pd.DataFrame([
    (c, int(rep_raw[c].notna().sum()), round(rep_raw[c].notna().mean() * 100, 1),
     round(date_share(rep_raw[c]) * 100, 1)) for c in cols
], columns=['Колонка отчёта', 'Заполнено строк', 'Заполнено, %', 'Похоже на дату, %'])
if EX_DATE_COLS:
    rep_date_cols = list(EX_DATE_COLS)
else:
    rep_date_cols = [c for c in cols if c not in (ex_id, ex_num)
                     and (any(k in _low(c) for k in DATE_WORDS) or date_share(rep_raw[c]) >= 0.9)
                     and date_share(rep_raw[c]) >= 0.5]
print('Отчёт: строка заголовка %s, строк %s' % (rep_hdr, len(rep_raw)))
print('Колонки отчёта: номер = %r | ID = %r' % (ex_num, ex_id))
print('Колонки отчёта с датами:', rep_date_cols)
display(rep_fill)

rep = pd.DataFrame({
    'num_key': col_or_none(rep_raw, ex_num, norm_agr),
    'rep_id': col_or_none(rep_raw, ex_id, norm_id),
    'Номер договора (отчёт)': col_or_none(rep_raw, ex_num),
})
REP_DATES = []
for c in rep_date_cols:
    name = 'Отчёт: %s' % c
    rep[name] = to_date(rep_raw[c])
    REP_DATES.append(name)
rep = rep[rep['num_key'].notna()].drop_duplicates('num_key').reset_index(drop=True)

mine_raw = pd.read_excel(MINE_FILE, sheet_name=MINE_SHEET, dtype=object)
mine = pd.DataFrame({
    'num_key': mine_raw['Номер договора'].map(norm_agr),
    'Альфа: открытие': to_date(mine_raw['Дата открытия договора']),
    'Альфа: закрытие': to_date(mine_raw['Дата закрытия договора']),
})
mine = mine[mine['num_key'].notna()].drop_duplicates('num_key').reset_index(drop=True)
ALPHA_DATES = ['Альфа: открытие', 'Альфа: закрытие']
print('Отчёт: договоров %s | моя выгрузка: %s' % (len(rep), len(mine)))


## 2. Сопоставление ЦФТ и отчёта

Сначала по номеру договора, затем оставшиеся - по ID. Заодно видно, совпадает ли `id` ЦФТ с ID договора в отчёте.


In [ ]:
by_num = cft.merge(rep, on='num_key', suffixes=('', ' (отчёт)'))
by_num['Как сопоставлен'] = 'по номеру'
id_pairs = by_num[by_num['cft_id'].notna() & by_num['rep_id'].notna()]
id_same = float((id_pairs['cft_id'] == id_pairs['rep_id']).mean() * 100) if len(id_pairs) else None

cft_rest = cft[~cft['num_key'].isin(set(by_num['num_key']))]
rep_rest = rep[~rep['num_key'].isin(set(by_num['num_key']))]
by_id = pd.DataFrame()
if ex_id and id_same is not None and id_same >= 80:
    by_id = (cft_rest.dropna(subset=['cft_id'])
             .merge(rep_rest.dropna(subset=['rep_id']).drop_duplicates('rep_id').rename(columns={'num_key': 'num_key_rep'}),
                    left_on='cft_id', right_on='rep_id'))
    by_id['Как сопоставлен'] = 'по ID (номер разный)'
pairs = pd.concat([by_num, by_id], ignore_index=True)

match_stats = pd.DataFrame([
    ('Договоров в ЦФТ', len(cft)),
    ('Договоров в отчёте за сентябрь', len(rep)),
    ('Сопоставлено по номеру', len(by_num)),
    ('  id ЦФТ = ID договора в отчёте, %', round(id_same, 1) if id_same is not None else 'нет ID в отчёте'),
    ('Сопоставлено по ID (номер разный)', len(by_id)),
    ('Только в ЦФТ', len(cft) - len(pairs)),
    ('Только в отчёте', len(rep) - len(pairs)),
], columns=['Сопоставление ЦФТ и отчёта', 'Значение'])
display(match_stats)


## 3. Сходятся ли даты

«B − A» - насколько дата второго источника позже первой. «Совпадают» - день в день, «±1 день» - с точностью до дня.


In [ ]:
def cmp_dates(df, a, b, src):
    fa, fb = df[a].notna(), df[b].notna()
    both = fa & fb
    d = (df.loc[both, b] - df.loc[both, a]).dt.days
    n = int(both.sum())
    return {
        'Источники': src, 'Дата A': a, 'Дата B': b, 'Договоров': len(df), 'Обе заполнены': n,
        'Совпадают, %': round(float((d == 0).mean() * 100), 1) if n else None,
        '±1 день, %': round(float((d.abs() <= 1).mean() * 100), 1) if n else None,
        'Медиана B − A, дней': float(d.median()) if n else None,
        'Только A заполнена': int((fa & ~fb).sum()),
        'Только B заполнена': int((fb & ~fa).sum()),
    }

rows = []
for r in REP_DATES:
    for c in CFT_DATES:
        rows.append(cmp_dates(pairs, c, r, 'ЦФТ - отчёт'))
cft_mine = cft.merge(mine, on='num_key')
for a in ALPHA_DATES:
    for c in CFT_DATES:
        rows.append(cmp_dates(cft_mine, c, a, 'ЦФТ - Альфа'))
rep_mine = rep.merge(mine, on='num_key')
for r in REP_DATES:
    for a in ALPHA_DATES:
        rows.append(cmp_dates(rep_mine, a, r, 'Альфа - отчёт'))
date_cmp = pd.DataFrame(rows)

best = (date_cmp.dropna(subset=['Совпадают, %'])
        .sort_values(['Дата B', 'Совпадают, %'], ascending=[True, False])
        .drop_duplicates(['Источники', 'Дата B'])
        [['Источники', 'Дата B', 'Дата A', 'Обе заполнены', 'Совпадают, %', '±1 день, %', 'Медиана B − A, дней']]
        .rename(columns={'Дата B': 'Дата', 'Дата A': 'Больше всего совпадает с'}))
display(best)
display(date_cmp)


## 4. Даты внутри ЦФТ: окончание, закрытие и статус

Правда ли, что дата закрытия всегда на полгода позже даты окончания, и как заполненность дат связана со статусом.


In [ ]:
lag = cft.dropna(subset=['ЦФТ: окончание', 'ЦФТ: закрытие'])
lag_m = ((lag['ЦФТ: закрытие'].dt.year - lag['ЦФТ: окончание'].dt.year) * 12
         + (lag['ЦФТ: закрытие'].dt.month - lag['ЦФТ: окончание'].dt.month))
close_lag = (lag_m.value_counts().rename_axis('Закрытие позже окончания, месяцев').reset_index(name='Договоров')
             .sort_values('Договоров', ascending=False).head(15))
print('Обе даты заполнены у %s договоров, ровно 6 месяцев: %s (%.1f%%)' % (
    len(lag), int((lag_m == 6).sum()), (lag_m == 6).mean() * 100 if len(lag) else 0))

def fill_kind(r):
    e, c = pd.notna(r['ЦФТ: окончание']), pd.notna(r['ЦФТ: закрытие'])
    if e and c:
        return 'окончание и закрытие'
    if e:
        return 'только окончание'
    if c:
        return 'только закрытие'
    return 'обе пустые'

cft['Даты окончания/закрытия'] = cft.apply(fill_kind, axis=1)
end_vs = cft['ЦФТ: окончание'].map(lambda d: 'пусто' if pd.isna(d) else ('раньше %s' % as_of_ru if d < as_of else 'не раньше %s' % as_of_ru))
status_dates = pd.crosstab(cft['Статус (ЦФТ)'].fillna('(пусто)'), cft['Даты окончания/закрытия'], margins=True, margins_name='Всего')
status_end = pd.crosstab(cft['Статус (ЦФТ)'].fillna('(пусто)'), end_vs.rename('Окончание'), margins=True, margins_name='Всего')
display(close_lag)
display(status_dates)
display(status_end)


## 5. Excel

In [ ]:
def autofit(ws):
    for col_ in ws.columns:
        width = max(len(str(c.value)) if c.value is not None else 0 for c in col_)
        ws.column_dimensions[col_[0].column_letter].width = min(max(width + 2, 10), 70)

def dates(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    return df

detail = pairs.merge(mine, on='num_key', how='left')
detail_cols = (['Как сопоставлен', 'Номер договора (ЦФТ)', 'Номер договора (отчёт)', 'Клиент (ЦФТ)', 'Статус (ЦФТ)']
               + CFT_DATES + REP_DATES + ALPHA_DATES)
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as w:
    row = 0
    for t in (match_stats, best, date_cmp, close_lag, rep_fill):
        t.to_excel(w, index=False, sheet_name='Итог', startrow=row)
        row += len(t) + 2
    row = 0
    for t in (status_dates, status_end):
        t.reset_index().to_excel(w, index=False, sheet_name='Статусы и даты ЦФТ', startrow=row)
        row += len(t) + 3
    dates(detail[detail_cols]).to_excel(w, index=False, sheet_name='Даты по договорам')
    for ws in w.book.worksheets:
        autofit(ws)
    w.book['Даты по договорам'].freeze_panes = 'A2'
print(OUT_FILE)
